# Digital Witness — YOLO26 + MobileNetV2 + LSTM Training Pipeline

**Final Year Project — Retail Shoplifting Detection**

This single notebook replaces all 25+ source files. Run cells top-to-bottom in Google Colab.

### Pipeline Architecture
```
Video Frame
    ↓
YOLO26 Detection + ByteTrack   (Cell 3 — fine-tune on your annotations)
    ↓
MobileNetV2 Feature Extraction  (Cell 4 — 1280→512 projected features)
    ↓
Bidirectional LSTM + Attention  (Cell 5 — 2-class: normal / shoplifting)
    ↓
Intent Scoring + Bias Assessment (Cell 8)
    ↓
Case File + Visualization        (Cell 9)
```

### Setup
1. Upload `yolo26n.pt` to `Drive/DigitalWitness/yolo26n.pt`
2. Place YOLO-format dataset at `Drive/DigitalWitness/dataset/` with `data.yaml`
3. Place training videos at `Drive/DigitalWitness/dataset/videos/normal/` and `.../shoplifting/`
4. Runtime → Change runtime type → **GPU**
5. Run all cells

In [ ]:
# ============================================================
# CELL 1 — Setup & Install
# ============================================================
!pip install ultralytics torch torchvision opencv-python-headless \
    numpy scikit-learn matplotlib plotly pandas reportlab Pillow -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify environment
import torch
import torchvision
from ultralytics import YOLO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device      : {device}')
print(f'PyTorch     : {torch.__version__}')
print(f'TorchVision : {torchvision.__version__}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
print('Ultralytics : OK')

In [ ]:
# ============================================================
# CELL 2 — Configuration
# ============================================================
from pathlib import Path
from datetime import datetime

# ---- Paths (Google Drive) ----
DRIVE_ROOT      = Path('/content/drive/MyDrive/DigitalWitness')
MODELS_DIR      = DRIVE_ROOT / 'models'
DATASET_YAML    = str(DRIVE_ROOT / 'dataset/data.yaml')
TRAIN_VIDEOS    = DRIVE_ROOT / 'dataset/videos'  # subdirs: normal/, shoplifting/
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ---- 2-class system (fixing the 4-class mismatch in original code) ----
BEHAVIOR_CLASSES = ['normal', 'shoplifting']
NUM_CLASSES      = 2

# ---- Model paths ----
YOLO26_BASE     = '/content/yolo26n.pt'               # local copy for fast I/O
YOLO26_RETAIL   = str(MODELS_DIR / 'yolo26_retail.pt')
MOBILENET_SAVE  = str(MODELS_DIR / 'mobilenet_extractor.pt')
LSTM_SAVE       = str(MODELS_DIR / 'lstm_classifier.pt')
LSTM_INFO_SAVE  = str(MODELS_DIR / 'lstm_classifier_info.json')

# ---- Detection (YOLO26 + ByteTrack) ----
YOLO_CONF           = 0.3
YOLO_IOU            = 0.45
PROXIMITY_THRESHOLD = 150   # pixels — consistent value (was 100 vs 150 mismatch)

# ---- MobileNetV2 feature extractor ----
MOBILENET_INPUT_SIZE  = (224, 224)
MOBILENET_FEATURE_DIM = 512   # projected from 1280

# ---- LSTM ----
LSTM_HIDDEN_DIM = 256
LSTM_NUM_LAYERS = 2
LSTM_DROPOUT    = 0.3
LSTM_SEQ_LEN    = 30   # frames per sequence (~1 sec at 30 fps)
LSTM_STRIDE     = 15   # 50% overlap between windows

# ---- Training hyperparameters ----
EPOCHS_YOLO     = 50
EPOCHS_MOBILENET = 20
EPOCHS_LSTM     = 50
BATCH_SIZE      = 16
LR_MOBILENET    = 1e-4   # lower LR for fine-tuning pretrained model
LR_LSTM         = 1e-3

# ---- Intent scoring thresholds ----
THRESHOLD_LOW      = 0.3
THRESHOLD_MEDIUM   = 0.5
THRESHOLD_HIGH     = 0.7
THRESHOLD_CRITICAL = 0.85

# ---- Scoring weights (must sum to 1.0) ----
W_CONCEALMENT = 0.50   # primary shoplifting signal (no POS in MVP)
W_BYPASS      = 0.35   # checkout avoidance
W_DURATION    = 0.15   # time in suspicious state

print('Configuration loaded:')
print(f'  Classes         : {BEHAVIOR_CLASSES}')
print(f'  Feature dim     : {MOBILENET_FEATURE_DIM}')
print(f'  Sequence length : {LSTM_SEQ_LEN} frames')
print(f'  Proximity thresh: {PROXIMITY_THRESHOLD} px')
print(f'  Models dir      : {MODELS_DIR}')

In [ ]:
# ============================================================
# CELL 3 — YOLO26 Fine-tuning
# ============================================================
# YOLO26 (Jan 2026, Ultralytics): NMS-free, 43% faster than YOLOv8.
# Same API as earlier YOLO versions — just load yolo26n.pt directly.
# ============================================================
import shutil
from ultralytics import YOLO

# Copy yolo26n.pt from Drive → local Colab storage (much faster training I/O)
yolo26_drive_path = DRIVE_ROOT / 'yolo26n.pt'
if yolo26_drive_path.exists():
    shutil.copy(str(yolo26_drive_path), YOLO26_BASE)
    print(f'Copied yolo26n.pt from Drive → {YOLO26_BASE}')
elif Path(YOLO26_BASE).exists():
    print(f'Using existing local {YOLO26_BASE}')
else:
    raise FileNotFoundError(
        'yolo26n.pt not found. Place it at Drive/DigitalWitness/yolo26n.pt'
    )

# ---- Load YOLO26 ----
model = YOLO(YOLO26_BASE)
print(f'Loaded YOLO26 from {YOLO26_BASE}')
print(f'Dataset YAML: {DATASET_YAML}')

# ---- Fine-tune on your YOLO-format annotations ----
# data.yaml format:
#   train: /content/drive/MyDrive/DigitalWitness/dataset/images/train
#   val:   /content/drive/MyDrive/DigitalWitness/dataset/images/val
#   nc: <number of classes>
#   names: [person, bottle, cup, bag, ...]   (retail-relevant COCO classes)
#
# freeze=10: freezes first 10 backbone layers, trains detection head only
print(f'\nStarting YOLO26 fine-tuning: {EPOCHS_YOLO} epochs, batch={BATCH_SIZE}, freeze=10')

train_results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS_YOLO,
    imgsz=640,
    batch=BATCH_SIZE,
    freeze=10,
    project=str(DRIVE_ROOT / 'runs'),
    name='yolo26_retail',
    save=True,
    patience=20,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True,
)

# ---- Save best weights to Drive models/ ----
best_weights = DRIVE_ROOT / 'runs/yolo26_retail/weights/best.pt'
if best_weights.exists():
    shutil.copy(str(best_weights), YOLO26_RETAIL)
    print(f'\nFine-tuned YOLO26 saved to: {YOLO26_RETAIL}')
else:
    print('Warning: best.pt not found, saving last.pt')
    last_weights = DRIVE_ROOT / 'runs/yolo26_retail/weights/last.pt'
    if last_weights.exists():
        shutil.copy(str(last_weights), YOLO26_RETAIL)

# ---- Display training metrics ----
print('\n=== YOLO26 Training Metrics ===')
try:
    metrics = train_results.results_dict
    print(f"  mAP50     : {metrics.get('metrics/mAP50(B)', 0):.3f}")
    print(f"  mAP50-95  : {metrics.get('metrics/mAP50-95(B)', 0):.3f}")
    print(f"  Precision : {metrics.get('metrics/precision(B)', 0):.3f}")
    print(f"  Recall    : {metrics.get('metrics/recall(B)', 0):.3f}")
except Exception as e:
    print(f'(Could not parse metrics: {e})')

In [ ]:
# ============================================================
# CELL 4 — MobileNetV2 Feature Extractor
# ============================================================
# Fixes from original code:
#   1. weights=MobileNet_V2_Weights.DEFAULT  (not deprecated pretrained=True)
#   2. model.features used directly          (not children()[:-1] which breaks MobileNetV2)
#   3. AdaptiveAvgPool2d added before Linear (was missing → 62720 ≠ 1280 crash)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights
import numpy as np
import cv2
import random

# ---- Model Definition ----
class MobileNetFeatureExtractor(nn.Module):
    '''
    MobileNetV2 feature extractor with 1280→512 projection.

    MobileNetV2 features[0..18]:
      0:    Conv (stride 2)
      1-18: InvertedResidual blocks + final Conv
    We freeze layers 0-14, unfreeze the last 3 InvertedResidual blocks
    (indices 15, 16, 17) and the final ConvBNActivation (index 18).
    '''
    def __init__(self, feature_dim=MOBILENET_FEATURE_DIM, device='auto'):
        super().__init__()
        self._dev = (torch.device('cuda' if torch.cuda.is_available() else 'cpu')
                     if device == 'auto' else torch.device(device))

        # Fix 1: use updated weights API (not deprecated pretrained=True)
        mobilenet = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

        # Fix 2: use model.features directly
        # Output shape: (batch, 1280, 7, 7) for 224x224 input
        self.features = mobilenet.features

        # Freeze early layers 0-14, unfreeze last 3 InvertedResidual + final Conv
        for i, layer in enumerate(self.features):
            requires_grad = (i >= 15)
            for param in layer.parameters():
                param.requires_grad = requires_grad

        # Fix 3: pool before linear (without pool → Flatten gives 1280*7*7=62720)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
        self.to(self._dev)

    def forward(self, x):
        x = self.features(x)    # (B, 1280, 7, 7)
        x = self.pool(x)        # (B, 1280, 1, 1)
        x = self.projection(x)  # (B, feature_dim)
        return x

    @torch.no_grad()
    def extract(self, rgb_frame_np):
        '''Extract 512-dim feature vector from a single RGB numpy frame.'''
        xform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(MOBILENET_INPUT_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
        x = xform(rgb_frame_np).unsqueeze(0).to(self._dev)
        self.eval()
        return self(x).cpu().numpy().flatten()

print('MobileNetFeatureExtractor defined')

# ---- Video Frame Dataset ----
class VideoFrameDataset(Dataset):
    def __init__(self, video_dir, classes, frames_per_video=100, augment=True):
        xform_list = [
            transforms.ToPILImage(),
            transforms.Resize(MOBILENET_INPUT_SIZE),
        ]
        if augment:
            xform_list += [
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            ]
        xform_list += [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ]
        self.transform = transforms.Compose(xform_list)
        self.data = []

        video_dir = Path(video_dir)
        for label, cls_name in enumerate(classes):
            cls_dir = video_dir / cls_name
            if not cls_dir.exists():
                print(f'Warning: {cls_dir} not found, skipping')
                continue
            videos = list(cls_dir.glob('*.mp4')) + list(cls_dir.glob('*.avi'))
            print(f'{cls_name}: {len(videos)} videos')
            for vpath in videos:
                cap = cv2.VideoCapture(str(vpath))
                total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                if total < 1:
                    cap.release(); continue
                idxs = sorted(random.sample(range(total), min(frames_per_video, total)))
                for idx in idxs:
                    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                    ret, frame = cap.read()
                    if ret:
                        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        self.data.append((rgb, label))
                cap.release()
        print(f'Total frames loaded: {len(self.data)}')

    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        frame, label = self.data[i]
        return self.transform(frame), label

# ---- Fine-tuning ----
print('\nLoading training frames for MobileNetV2 fine-tuning...')
dataset = VideoFrameDataset(str(TRAIN_VIDEOS), BEHAVIOR_CLASSES, frames_per_video=100)

if len(dataset) < 10:
    print('WARNING: Very few training frames found.')
    print(f'Expected videos at: {TRAIN_VIDEOS}/normal/ and {TRAIN_VIDEOS}/shoplifting/')
    feature_extractor = MobileNetFeatureExtractor()   # still define for later cells
else:
    n_train = int(0.8 * len(dataset))
    n_val   = len(dataset) - n_train
    train_ds, val_ds = torch.utils.data.random_split(dataset, [n_train, n_val])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=2)

    feature_extractor = MobileNetFeatureExtractor()
    trainable = [p for p in feature_extractor.parameters() if p.requires_grad]
    print(f'Trainable params: {sum(p.numel() for p in trainable):,}')

    optimizer  = optim.Adam(trainable, lr=LR_MOBILENET)
    criterion  = nn.CrossEntropyLoss()
    scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)

    best_val_acc = 0.0
    print(f'Training {EPOCHS_MOBILENET} epochs on {n_train} frames...')

    for epoch in range(EPOCHS_MOBILENET):
        # Train
        feature_extractor.train()
        t_loss = t_correct = t_total = 0
        for X, y in train_loader:
            X, y = X.to(feature_extractor._dev), y.to(feature_extractor._dev)
            optimizer.zero_grad()
            out  = feature_extractor(X)
            loss = criterion(out, y)
            loss.backward(); optimizer.step()
            t_loss    += loss.item()
            t_correct += (out.argmax(1) == y).sum().item()
            t_total   += y.size(0)

        # Validate
        feature_extractor.eval()
        v_correct = v_total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(feature_extractor._dev), y.to(feature_extractor._dev)
                out = feature_extractor(X)
                v_correct += (out.argmax(1) == y).sum().item()
                v_total   += y.size(0)

        train_acc = t_correct / t_total
        val_acc   = v_correct / v_total
        scheduler.step()

        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch+1:3d}/{EPOCHS_MOBILENET}: '
                  f'Loss={t_loss/len(train_loader):.4f} '
                  f'TrainAcc={train_acc:.3f} ValAcc={val_acc:.3f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(feature_extractor.state_dict(), MOBILENET_SAVE)

    feature_extractor.load_state_dict(torch.load(MOBILENET_SAVE))
    feature_extractor.eval()
    print(f'\nBest val accuracy: {best_val_acc:.1%}')
    print(f'Feature extractor saved → {MOBILENET_SAVE}')

In [ ]:
# ============================================================
# CELL 5 — LSTM Classifier (Bidirectional + Attention)
# ============================================================
import json
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_score,
                              recall_score, f1_score)
from sklearn.model_selection import train_test_split

# ---- Model Definition ----
class LSTMAttentionClassifier(nn.Module):
    '''
    Bidirectional LSTM with attention for temporal behavior classification.

    Architecture:
      Input (B, T, 512)
        → Bidirectional LSTM  → (B, T, 512)
        → Attention mechanism → context (B, 512)
        → FC classifier       → (B, num_classes)

    Attention learns which frames in the sequence matter most for classification.
    Bidirectional LSTM captures both past and future context.
    '''
    def __init__(self, input_dim=MOBILENET_FEATURE_DIM, hidden_dim=LSTM_HIDDEN_DIM,
                 num_layers=LSTM_NUM_LAYERS, num_classes=NUM_CLASSES,
                 dropout=LSTM_DROPOUT, bidirectional=True):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        attn_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.attention = nn.Sequential(
            nn.Linear(attn_dim, attn_dim // 2),
            nn.Tanh(),
            nn.Linear(attn_dim // 2, 1)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(attn_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)               # (B, T, attn_dim)
        attn_w = self.attention(lstm_out)         # (B, T, 1)
        attn_w = torch.softmax(attn_w, dim=1)
        context = torch.sum(attn_w * lstm_out, dim=1)  # (B, attn_dim)
        logits  = self.classifier(context)        # (B, num_classes)
        return logits, attn_w.squeeze(-1)

print('LSTMAttentionClassifier defined')

# ---- Extract CNN feature sequences from training videos ----
def extract_sequences(video_dir, classes, extractor, seq_len=LSTM_SEQ_LEN, stride=LSTM_STRIDE):
    '''Extract sliding-window feature sequences from all training videos.'''
    X, y = [], []
    video_dir = Path(video_dir)
    for label, cls_name in enumerate(classes):
        cls_dir = video_dir / cls_name
        if not cls_dir.exists():
            print(f'Warning: {cls_dir} not found'); continue
        videos = list(cls_dir.glob('*.mp4')) + list(cls_dir.glob('*.avi'))
        print(f'{cls_name}: {len(videos)} videos')
        for vpath in videos:
            cap = cv2.VideoCapture(str(vpath))
            frame_feats = []
            while True:
                ret, frame = cap.read()
                if not ret: break
                rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                feat = extractor.extract(rgb)
                frame_feats.append(feat)
            cap.release()
            if len(frame_feats) < seq_len:
                continue
            arr = np.array(frame_feats)  # (T, 512)
            for start in range(0, len(frame_feats) - seq_len + 1, stride):
                X.append(arr[start:start + seq_len])
                y.append(label)
    return np.array(X), np.array(y)

print('\nExtracting MobileNetV2 feature sequences from training videos...')
feature_extractor.eval()
X_all, y_all = extract_sequences(str(TRAIN_VIDEOS), BEHAVIOR_CLASSES, feature_extractor)

print(f'\nTotal sequences: {len(X_all)}')
for i, cls in enumerate(BEHAVIOR_CLASSES):
    print(f'  {cls}: {np.sum(y_all == i)} sequences')

# ---- Train LSTM ----
if len(X_all) < 10:
    print('ERROR: Not enough sequences. Place training videos at:')
    print(f'  {TRAIN_VIDEOS}/normal/ and {TRAIN_VIDEOS}/shoplifting/')
else:
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
    )
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    tr_ds  = torch.utils.data.TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr))
    val_ds = torch.utils.data.TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))
    tr_loader  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    lstm_model = LSTMAttentionClassifier().to(dev)
    optimizer  = optim.Adam(lstm_model.parameters(), lr=LR_LSTM)
    criterion  = nn.CrossEntropyLoss()
    scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, verbose=True)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc   = 0.0
    patience_count = 0
    PATIENCE       = 10

    print(f'Training LSTM: {len(X_tr)} train, {len(X_val)} val sequences')

    for epoch in range(EPOCHS_LSTM):
        # Train
        lstm_model.train()
        t_loss = t_correct = t_total = 0
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(dev), yb.to(dev)
            optimizer.zero_grad()
            logits, _ = lstm_model(Xb)
            loss = criterion(logits, yb)
            loss.backward(); optimizer.step()
            t_loss    += loss.item()
            t_correct += (logits.argmax(1) == yb).sum().item()
            t_total   += yb.size(0)

        # Validate
        lstm_model.eval()
        v_loss = v_correct = v_total = 0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(dev), yb.to(dev)
                logits, _ = lstm_model(Xb)
                loss = criterion(logits, yb)
                v_loss    += loss.item()
                v_correct += (logits.argmax(1) == yb).sum().item()
                v_total   += yb.size(0)

        t_acc = t_correct / t_total
        v_acc = v_correct / v_total
        v_loss_avg = v_loss / len(val_loader)
        scheduler.step(v_loss_avg)

        history['train_loss'].append(t_loss / len(tr_loader))
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss_avg)
        history['val_acc'].append(v_acc)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}/{EPOCHS_LSTM}: '
                  f'TrainAcc={t_acc:.3f}  ValAcc={v_acc:.3f}')

        if v_acc > best_val_acc:
            best_val_acc   = v_acc
            patience_count = 0
            torch.save({'state_dict': lstm_model.state_dict(),
                        'config': {'input_dim': MOBILENET_FEATURE_DIM,
                                   'hidden_dim': LSTM_HIDDEN_DIM,
                                   'num_layers': LSTM_NUM_LAYERS,
                                   'num_classes': NUM_CLASSES,
                                   'classes': BEHAVIOR_CLASSES}},
                       LSTM_SAVE)
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f'Early stopping at epoch {epoch+1}')
                break

    # Load best weights
    ckpt = torch.load(LSTM_SAVE)
    lstm_model.load_state_dict(ckpt['state_dict'])
    lstm_model.eval()

    # ---- Evaluation ----
    all_preds, all_labels = [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            logits, _ = lstm_model(Xb.to(dev))
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(yb.tolist())

    print('\n=== LSTM Evaluation ===')
    print(classification_report(all_labels, all_preds, target_names=BEHAVIOR_CLASSES))

    # Plots
    cm = confusion_matrix(all_labels, all_preds)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    im = axes[0].imshow(cm, cmap='Blues')
    axes[0].set_xticks(range(NUM_CLASSES)); axes[0].set_xticklabels(BEHAVIOR_CLASSES)
    axes[0].set_yticks(range(NUM_CLASSES)); axes[0].set_yticklabels(BEHAVIOR_CLASSES)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            axes[0].text(j, i, str(cm[i, j]), ha='center', va='center',
                         fontsize=14, color='white' if cm[i,j] > cm.max()/2 else 'black')
    axes[0].set_title('Confusion Matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

    axes[1].plot(history['train_acc'], label='Train Acc')
    axes[1].plot(history['val_acc'],   label='Val Acc')
    axes[1].set_title('Training History'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy'); axes[1].legend()

    plt.tight_layout()
    plt.savefig(str(MODELS_DIR / 'lstm_training.png'), dpi=120, bbox_inches='tight')
    plt.show()

    # Save metrics
    info = {
        'training_date'      : datetime.now().isoformat(),
        'classes'            : BEHAVIOR_CLASSES,
        'final_train_acc'    : history['train_acc'][-1],
        'final_val_acc'      : best_val_acc,
        'n_train_samples'    : len(X_tr),
        'n_val_samples'      : len(X_val),
        'input_dim'          : MOBILENET_FEATURE_DIM,
        'sequence_length'    : LSTM_SEQ_LEN,
        'stride'             : LSTM_STRIDE,
        'epochs'             : len(history['train_acc']),
        'confusion_matrix'   : cm.tolist(),
        'metrics': {
            'accuracy' : accuracy_score(all_labels, all_preds),
            'precision': precision_score(all_labels, all_preds, average='weighted', zero_division=0),
            'recall'   : recall_score(all_labels, all_preds, average='weighted', zero_division=0),
            'f1_score' : f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        },
        'per_class_metrics': {
            'precision': {c: precision_score(all_labels, all_preds, average=None, zero_division=0)[i]
                          for i, c in enumerate(BEHAVIOR_CLASSES)},
            'recall'   : {c: recall_score(all_labels, all_preds, average=None, zero_division=0)[i]
                          for i, c in enumerate(BEHAVIOR_CLASSES)},
            'f1'       : {c: f1_score(all_labels, all_preds, average=None, zero_division=0)[i]
                          for i, c in enumerate(BEHAVIOR_CLASSES)},
        }
    }
    with open(LSTM_INFO_SAVE, 'w') as f:
        json.dump(info, f, indent=2)

    print(f'\nBest validation accuracy : {best_val_acc:.1%}')
    print(f'LSTM model saved        → {LSTM_SAVE}')
    print(f'Metrics saved           → {LSTM_INFO_SAVE}')

In [ ]:
# ============================================================
# CELL 6 — Object Tracker (ByteTrack via YOLO26 built-in)
# ============================================================
# Person-product interaction detection:
#   approach : distance < PROXIMITY_THRESHOLD (150 px)
#   pickup   : product bbox overlaps person bbox 30–80%
#   hold     : product mostly inside person bbox (>80%), upper body
#   conceal  : product mostly inside person bbox (>80%), lower body
# ============================================================

PRODUCT_CLASSES = {
    'bottle', 'cup', 'bowl', 'banana', 'apple', 'sandwich',
    'backpack', 'handbag', 'book', 'cell phone', 'orange', 'donut'
}

def detect_interactions(video_path, yolo_path, conf=YOLO_CONF, iou=YOLO_IOU,
                        prox=PROXIMITY_THRESHOLD):
    '''
    Run YOLO26 with ByteTrack and detect person-product interactions.
    Returns list of interaction dicts with timestamp, type, confidence.
    '''
    model = YOLO(yolo_path)
    cap   = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0

    interactions    = []
    persons_seen    = set()
    products_seen   = set()
    frame_num       = 0

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_num += 1
        ts = frame_num / fps

        results = model.track(frame, conf=conf, iou=iou, persist=True, verbose=False)[0]
        if results.boxes is None or len(results.boxes) == 0:
            continue

        persons  = []
        products = []
        boxes    = results.boxes
        ids      = boxes.id.int().tolist() if boxes.id is not None else [-1] * len(boxes)

        for box, cls_id, tid, conf_val in zip(boxes.xyxy, boxes.cls, ids, boxes.conf):
            x1, y1, x2, y2 = map(int, box.tolist())
            cls_name = model.names[int(cls_id)]
            if cls_name == 'person':
                persons.append({'bbox': (x1,y1,x2,y2), 'id': tid, 'conf': float(conf_val)})
                persons_seen.add(tid)
            elif cls_name in PRODUCT_CLASSES:
                products.append({'bbox': (x1,y1,x2,y2), 'cls': cls_name, 'id': tid, 'conf': float(conf_val)})
                products_seen.add(cls_name)

        for person in persons:
            px1, py1, px2, py2 = person['bbox']
            pc = ((px1+px2)/2.0, (py1+py2)/2.0)

            for prod in products:
                bx1, by1, bx2, by2 = prod['bbox']
                bc = ((bx1+bx2)/2.0, (by1+by2)/2.0)

                dist = np.sqrt((pc[0]-bc[0])**2 + (pc[1]-bc[1])**2)

                # Overlap: how much of the product bbox is inside person bbox
                ix1 = max(px1, bx1); iy1 = max(py1, by1)
                ix2 = min(px2, bx2); iy2 = min(py2, by2)
                prod_area = max(1, (bx2-bx1) * (by2-by1))
                overlap   = max(0.0, (ix2-ix1)*(iy2-iy1)) / prod_area if ix2>ix1 and iy2>iy1 else 0.0

                # Classify interaction
                itype = None
                if overlap > 0.8:
                    # Product mostly inside person — hold (upper) or conceal (lower)
                    person_mid_y = (py1 + py2) / 2.0
                    itype = 'hold' if bc[1] < person_mid_y else 'conceal'
                elif overlap > 0.3:
                    itype = 'pickup'
                elif dist < prox:
                    itype = 'approach'

                if itype:
                    base_conf = (person['conf'] + prod['conf']) / 2.0
                    conf_adj  = 1.0 if overlap > 0.5 else 0.8 if overlap > 0.2 else 0.6
                    conf_adj *= 1.0 if dist < 50 else 0.85 if dist < 100 else 0.7
                    interactions.append({
                        'timestamp'  : ts,
                        'frame'      : frame_num,
                        'person_id'  : person['id'],
                        'product_cls': prod['cls'],
                        'type'       : itype,
                        'confidence' : base_conf * conf_adj,
                        'distance'   : float(dist),
                        'overlap'    : float(overlap)
                    })

    cap.release()
    return {
        'interactions'   : interactions,
        'persons_tracked': len(persons_seen),
        'products_seen'  : len(products_seen),
        'total_frames'   : frame_num,
    }

print('Tracker (detect_interactions) defined')
print(f'Proximity threshold: {PROXIMITY_THRESHOLD} px  (fixed — was 100 vs 150 mismatch)')
print(f'Tracked product classes: {sorted(PRODUCT_CLASSES)}')

In [ ]:
# ============================================================
# CELL 7 — Full Inference Pipeline
# ============================================================
# YOLO26 detection → MobileNetV2 features → LSTM classification
# Sliding window temporal analysis + weighted voting aggregation
# ============================================================

def run_inference(video_path,
                  yolo_path=YOLO26_RETAIL,
                  mobilenet_path=MOBILENET_SAVE,
                  lstm_path=LSTM_SAVE,
                  seq_len=LSTM_SEQ_LEN,
                  stride=LSTM_STRIDE,
                  frame_step=1,
                  progress_cb=None):
    '''
    Full inference pipeline on a video file.

    Returns dict with:
      predictions     : list of per-window LSTM predictions
      overall_class   : 'normal' or 'shoplifting'
      overall_conf    : confidence of overall classification
      is_shoplifting  : bool
      detections      : detection stats
      behavior_events : list of behavior event dicts
    '''
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ---- Load models ----
    yolo_model = YOLO(yolo_path)

    feat_model = MobileNetFeatureExtractor()
    feat_model.load_state_dict(torch.load(mobilenet_path, map_location=dev))
    feat_model.eval()

    ckpt = torch.load(lstm_path, map_location=dev)
    lstm = LSTMAttentionClassifier(
        num_classes=len(ckpt['config']['classes'])
    ).to(dev)
    lstm.load_state_dict(ckpt['state_dict'])
    lstm.eval()
    classes = ckpt['config']['classes']

    # ---- Video processing ----
    cap         = cv2.VideoCapture(video_path)
    fps         = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_f     = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    all_feats   = []
    persons_set = set()
    products_set = set()
    frame_num   = 0

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_num += 1
        if frame_num % frame_step != 0: continue

        if progress_cb and frame_num % 30 == 0:
            progress_cb(frame_num / total_f * 0.75,
                        f'Processing frame {frame_num}/{total_f}...')

        # YOLO26 detection with ByteTrack
        results = yolo_model.track(frame, conf=YOLO_CONF, iou=YOLO_IOU,
                                   persist=True, verbose=False)[0]
        if results.boxes is not None:
            ids = (results.boxes.id.int().tolist()
                   if results.boxes.id is not None else [-1]*len(results.boxes))
            for box, cls_id, tid in zip(results.boxes.xyxy, results.boxes.cls, ids):
                cls_name = yolo_model.names[int(cls_id)]
                if cls_name == 'person':
                    persons_set.add(tid)
                elif cls_name in PRODUCT_CLASSES:
                    products_set.add(cls_name)

        # MobileNetV2 feature extraction
        rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        feat = feat_model.extract(rgb)
        all_feats.append(feat)

    cap.release()
    duration = frame_num / fps

    if progress_cb:
        progress_cb(0.8, 'Running LSTM temporal classification...')

    # ---- Sliding-window LSTM classification ----
    feats_arr   = np.array(all_feats)  # (N_frames, 512)
    predictions = []

    if len(all_feats) < seq_len:
        # Pad short video
        pad = np.zeros((seq_len - len(all_feats), MOBILENET_FEATURE_DIM))
        seq = np.vstack([feats_arr, pad])
        x   = torch.FloatTensor(seq[np.newaxis]).to(dev)
        with torch.no_grad():
            logits, _ = lstm(x)
            probs     = torch.softmax(logits, dim=1).cpu().numpy()[0]
        predictions.append({
            'start'        : 0.0,
            'end'          : duration,
            'class'        : classes[int(np.argmax(probs))],
            'confidence'   : float(np.max(probs)) * (len(all_feats) / seq_len),
            'probabilities': {c: float(p) for c, p in zip(classes, probs)}
        })
    else:
        for start in range(0, len(all_feats) - seq_len + 1, stride):
            seq = feats_arr[start:start + seq_len]
            x   = torch.FloatTensor(seq[np.newaxis]).to(dev)
            with torch.no_grad():
                logits, _ = lstm(x)
                probs     = torch.softmax(logits, dim=1).cpu().numpy()[0]
            predictions.append({
                'start'        : start / fps,
                'end'          : (start + seq_len) / fps,
                'class'        : classes[int(np.argmax(probs))],
                'confidence'   : float(np.max(probs)),
                'probabilities': {c: float(p) for c, p in zip(classes, probs)}
            })

    # ---- Weighted aggregation (2x weight for suspicious) ----
    shoplifting_preds = [p for p in predictions if p['class'] == 'shoplifting']
    normal_preds      = [p for p in predictions if p['class'] == 'normal']
    shop_score = sum(p['confidence'] for p in shoplifting_preds) * 2.0
    norm_score = sum(p['confidence'] for p in normal_preds)
    total_score = shop_score + norm_score

    if total_score > 0 and shop_score / total_score > 0.3 and shoplifting_preds:
        best_shop     = max(shoplifting_preds, key=lambda p: p['confidence'])
        overall_class = 'shoplifting'
        overall_conf  = best_shop['confidence']
    else:
        overall_class = 'normal'
        overall_conf  = (np.mean([p['confidence'] for p in normal_preds])
                         if normal_preds else 0.5)

    if progress_cb:
        progress_cb(1.0, 'Done!')

    # Convert predictions to behavior_events format for scoring
    behavior_events = [
        {
            'behavior_type': p['class'],
            'start_time'   : p['start'],
            'end_time'     : p['end'],
            'confidence'   : p['confidence'],
            'probabilities': p['probabilities']
        }
        for p in predictions
    ]

    return {
        'success'        : True,
        'predictions'    : predictions,
        'overall_class'  : overall_class,
        'overall_conf'   : overall_conf,
        'is_shoplifting' : overall_class == 'shoplifting',
        'behavior_events': behavior_events,
        'duration'       : duration,
        'fps'            : fps,
        'total_frames'   : frame_num,
        'detections': {
            'persons_tracked'  : len(persons_set),
            'products_detected': len(products_set),
            'frames_processed' : frame_num,
        }
    }

print('run_inference() defined')
print('Usage: result = run_inference(\'path/to/video.mp4\')')

In [ ]:
# ============================================================
# CELL 8 — Analysis & Scoring
# ============================================================
# Intent scorer + bias-aware scoring + alert generation + edge cases
# ============================================================

def calculate_intent_score(behavior_events, video_duration):
    '''
    Video-only intent scoring (no POS data in MVP mode).

    Components:
      concealment/shoplifting : 50%  (primary signal)
      bypass                  : 35%  (checkout avoidance)
      duration                : 15%  (time in suspicious state)

    Returns dict with score (0-1), severity, components, explanation.
    '''
    suspicious_types = {'shoplifting', 'concealment', 'bypass'}
    conceal_events   = [e for e in behavior_events
                        if e['behavior_type'] in {'shoplifting', 'concealment'}]
    bypass_events    = [e for e in behavior_events if e['behavior_type'] == 'bypass']
    all_suspicious   = [e for e in behavior_events if e['behavior_type'] in suspicious_types]

    # Concealment score: avg_conf * min(count/3, 1)
    if conceal_events:
        avg_conf = np.mean([e['confidence'] for e in conceal_events])
        count_f  = min(1.0, len(conceal_events) / 3.0)
        s_conceal = avg_conf * count_f
    else:
        s_conceal = 0.0

    # Bypass score: max confidence (single bypass attempt is enough)
    s_bypass = max((e['confidence'] for e in bypass_events), default=0.0)

    # Duration score: fraction of video spent in suspicious state
    suspicious_secs = sum(e['end_time'] - e['start_time'] for e in all_suspicious)
    s_duration = min(1.0, (suspicious_secs / max(1.0, video_duration)) * 3.0) if all_suspicious else 0.0

    # Weighted sum
    total = W_CONCEALMENT * s_conceal + W_BYPASS * s_bypass + W_DURATION * s_duration
    total = max(0.0, min(1.0, total))

    # Severity
    if total < THRESHOLD_LOW:      severity = 'NONE'
    elif total < THRESHOLD_MEDIUM: severity = 'LOW'
    elif total < THRESHOLD_HIGH:   severity = 'MEDIUM'
    elif total < THRESHOLD_CRITICAL: severity = 'HIGH'
    else:                           severity = 'CRITICAL'

    explanation_parts = []
    if conceal_events:
        explanation_parts.append(f'{len(conceal_events)} concealment/shoplifting event(s) detected')
    if bypass_events:
        explanation_parts.append(f'{len(bypass_events)} checkout bypass event(s) detected')
    if not explanation_parts:
        explanation_parts.append('No significant suspicious activity detected')

    return {
        'score'      : total,
        'severity'   : severity,
        'explanation': '; '.join(explanation_parts) + f'. Score: {total:.2f} ({severity})',
        'components' : {
            'concealment': s_conceal,
            'bypass'     : s_bypass,
            'duration'   : s_duration,
        }
    }


def bias_aware_adjustment(intent_score_dict, quality_score=1.0):
    '''
    Apply fairness adjustments to the intent score.
    Conservative when video quality is low or analysis is uncertain.

    Returns:
      adjusted_score, fairness_score, requires_review, flags
    '''
    raw_score     = intent_score_dict['score']
    flags         = []
    adj_factor    = 1.0

    # Low video quality -> reduce confidence
    if quality_score < 0.5:
        adj_factor = min(adj_factor, 0.7)
        flags.append('Low video quality reduces analysis confidence')
    elif quality_score < 0.75:
        adj_factor = min(adj_factor, 0.85)
        flags.append('Moderate video quality concern')

    # High raw score: apply softened adjustment (preserve LSTM signal)
    if raw_score >= 0.5:
        adj_factor = adj_factor ** 0.5   # soften: 0.7 → 0.84

    adjusted_score = max(0.0, min(1.0, raw_score * adj_factor))
    fairness_score = quality_score * (1.0 - 0.2 * len(flags))
    requires_review = len(flags) > 0 or raw_score > THRESHOLD_HIGH

    return {
        'raw_score'      : raw_score,
        'adjusted_score' : adjusted_score,
        'fairness_score' : max(0.0, fairness_score),
        'adj_factor'     : adj_factor,
        'flags'          : flags,
        'requires_review': requires_review,
        'analysis_reliable': quality_score >= 0.4,
    }


def generate_alert(intent_score_dict, bias_result, behavior_events):
    '''
    Generate an alert dict if score exceeds threshold.
    All alerts require human validation — this is advisory only.
    '''
    score    = bias_result['adjusted_score']
    severity = intent_score_dict['severity']

    if score < THRESHOLD_MEDIUM:
        return None

    alert_id = f"ALERT-{datetime.now().strftime('%Y%m%d%H%M%S')}-0001"
    n_suspicious = sum(1 for e in behavior_events
                       if e['behavior_type'] in {'shoplifting', 'concealment', 'bypass'})

    message = (
        f'{n_suspicious} suspicious behavior event(s) detected. '
        f'Adjusted risk score: {score:.2f} ({severity}). '
        f'Human review required before any action.'
    )
    return {
        'alert_id'            : alert_id,
        'timestamp'           : datetime.now().isoformat(),
        'level'               : severity,
        'score'               : score,
        'message'             : message,
        'requires_human_review': True,
        'fairness_score'      : bias_result['fairness_score'],
    }


def check_edge_cases(behavior_events, video_duration, quality_score=1.0):
    '''
    Detect edge cases that may affect reliability.
    '''
    flags = []
    if not behavior_events:
        flags.append({'type': 'no_data', 'severity': 'medium',
                      'msg': 'No behavior events detected'})

    if quality_score < THRESHOLD_LOW:
        flags.append({'type': 'poor_quality', 'severity': 'high',
                      'msg': f'Video quality below minimum: {quality_score:.2f}'})

    # Check for ambiguous LSTM predictions (top two probabilities are close)
    ambiguous = 0
    for e in behavior_events:
        probs = sorted(e['probabilities'].values(), reverse=True)
        if len(probs) >= 2 and (probs[0] - probs[1]) < 0.2:
            ambiguous += 1
    if ambiguous > len(behavior_events) * 0.5:
        flags.append({'type': 'ambiguous', 'severity': 'medium',
                      'msg': f'{ambiguous} ambiguous LSTM predictions'})

    reliability = quality_score - 0.1 * sum(
        1 for f in flags if f['severity'] == 'high'
    ) - 0.05 * sum(
        1 for f in flags if f['severity'] == 'medium'
    )
    requires_review = any(f['severity'] == 'high' for f in flags)

    return {
        'flags'          : flags,
        'reliability'    : max(0.0, min(1.0, reliability)),
        'requires_review': requires_review,
    }


# ---- Quick demo of the scoring pipeline ----
demo_events = [
    {'behavior_type': 'shoplifting', 'start_time': 5.0, 'end_time': 8.0, 'confidence': 0.82,
     'probabilities': {'normal': 0.18, 'shoplifting': 0.82}},
    {'behavior_type': 'normal',      'start_time': 0.0, 'end_time': 5.0, 'confidence': 0.91,
     'probabilities': {'normal': 0.91, 'shoplifting': 0.09}},
]
demo_intent  = calculate_intent_score(demo_events, video_duration=30.0)
demo_bias    = bias_aware_adjustment(demo_intent, quality_score=0.85)
demo_edge    = check_edge_cases(demo_events, video_duration=30.0, quality_score=0.85)
demo_alert   = generate_alert(demo_intent, demo_bias, demo_events)

print('=== Demo Scoring Output ===')
print(f'Intent score  : {demo_intent["score"]:.3f}  ({demo_intent["severity"]})')
print(f'Adjusted score: {demo_bias["adjusted_score"]:.3f}')
print(f'Fairness score: {demo_bias["fairness_score"]:.1%}')
print(f'Alert         : {demo_alert["level"] if demo_alert else "None"}')
print(f'Edge cases    : {len(demo_edge["flags"])} flags')
print('All scoring functions ready.')

In [ ]:
# ============================================================
# CELL 9 — Output & Visualization
# ============================================================
# Case file builder, behavior timeline, results display
# ============================================================
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML

CASE_DIR = DRIVE_ROOT / 'cases'
CASE_DIR.mkdir(parents=True, exist_ok=True)


def build_case_file(video_path, inference_result, intent_score_dict,
                    bias_result, edge_result, alert):
    '''
    Assemble a complete JSON case file from all analysis outputs.
    Serves as the full audit trail for human review.
    '''
    case_id  = f"CASE-{datetime.now().strftime('%Y%m%d%H%M%S')}-0001"
    case_data = {
        'case_id'     : case_id,
        'created_at'  : datetime.now().isoformat(),
        'system'      : 'Digital Witness — YOLO26 + MobileNetV2 + LSTM',
        'video': {
            'path'    : str(video_path),
            'duration': inference_result.get('duration', 0),
            'fps'     : inference_result.get('fps', 0),
            'frames'  : inference_result.get('total_frames', 0),
        },
        'lstm_detection': {
            'classification': inference_result['overall_class'],
            'confidence'    : inference_result['overall_conf'],
            'is_shoplifting': inference_result['is_shoplifting'],
        },
        'detections'     : inference_result.get('detections', {}),
        'behavior_events': inference_result.get('behavior_events', []),
        'intent_score'   : intent_score_dict,
        'bias_assessment': bias_result,
        'edge_cases'     : edge_result,
        'alert'          : alert,
        'advisory_note'  : (
            'This is an advisory system. All alerts require human validation. '
            'This output does NOT determine guilt.'
        )
    }
    case_path = CASE_DIR / f'{case_id}.json'
    with open(case_path, 'w') as f:
        json.dump(case_data, f, indent=2)
    return case_id, str(case_path), case_data


def plot_behavior_timeline(behavior_events, duration, title='Behavior Timeline'):
    '''Plot a Gantt-style behavior timeline.'''
    colors = {
        'normal'     : '#2ecc71',
        'shoplifting': '#e74c3c',
        'concealment': '#e67e22',
        'bypass'     : '#c0392b',
    }
    fig, ax = plt.subplots(figsize=(14, 3))
    for i, event in enumerate(behavior_events):
        btype = event['behavior_type']
        start = event['start_time']
        width = event['end_time'] - start
        color = colors.get(btype, '#95a5a6')
        ax.barh(0, width, left=start, height=0.6, color=color,
                alpha=0.8, edgecolor='white', linewidth=0.5)

    # Legend
    patches = [mpatches.Patch(color=c, label=k) for k, c in colors.items()
               if any(e['behavior_type'] == k for e in behavior_events)]
    ax.legend(handles=patches, loc='upper right', fontsize=9)
    ax.set_xlim(0, max(duration, 1))
    ax.set_xlabel('Time (seconds)', fontsize=11)
    ax.set_yticks([])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.spines['left'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()
    return fig


def display_results(inference_result, intent_score_dict, bias_result, alert=None):
    '''Print a formatted results summary.'''
    sep = '=' * 60
    cls = inference_result['overall_class'].upper()
    conf = inference_result['overall_conf']
    score = intent_score_dict['score']
    severity = intent_score_dict['severity']
    adj = bias_result['adjusted_score']
    fairness = bias_result['fairness_score']

    print(sep)
    print('  DIGITAL WITNESS — ANALYSIS RESULTS')
    print(sep)
    print(f'  LSTM Classification : {cls}')
    print(f'  Confidence          : {conf:.1%}')
    print(f'  Intent Score (raw)  : {score:.3f}  [{severity}]')
    print(f'  Adjusted Score      : {adj:.3f}')
    print(f'  Fairness Score      : {fairness:.1%}')

    det = inference_result.get('detections', {})
    print(f'\n  Persons tracked     : {det.get("persons_tracked", 0)}')
    print(f'  Products detected   : {det.get("products_detected", 0)}')
    print(f'  Frames processed    : {det.get("frames_processed", 0)}')

    if alert:
        print(f'\n  *** ALERT: {alert["alert_id"]} ***')
        print(f'  Level   : {alert["level"]}')
        print(f'  Message : {alert["message"]}')
    else:
        print('\n  No alert generated.')

    print(f'\n  NOTE: This is an advisory system.')
    print(f'        Final decisions MUST be made by human operators.')
    print(sep)


def analyze_video(video_path,
                  yolo_path=YOLO26_RETAIL,
                  mobilenet_path=MOBILENET_SAVE,
                  lstm_path=LSTM_SAVE,
                  quality_score=1.0,
                  frame_step=1):
    '''
    End-to-end analysis: load video → inference → scoring → output.

    Args:
        video_path    : path to video file
        quality_score : 0-1 estimate of video quality (1.0 = good)
        frame_step    : process every Nth frame (3 = 3x faster, less accurate)

    Returns:
        case_id, case_path, full_results_dict
    '''
    print(f'Analyzing: {video_path}')

    # Step 1: Inference
    inference = run_inference(video_path, yolo_path, mobilenet_path, lstm_path,
                              frame_step=frame_step)

    # Step 2: Intent scoring
    intent = calculate_intent_score(inference['behavior_events'], inference['duration'])

    # Step 3: Bias-aware adjustment
    bias = bias_aware_adjustment(intent, quality_score=quality_score)

    # Step 4: Edge case detection
    edge = check_edge_cases(inference['behavior_events'], inference['duration'], quality_score)

    # Step 5: Alert generation
    alert = generate_alert(intent, bias, inference['behavior_events'])

    # Step 6: Build & save case file
    case_id, case_path, case_data = build_case_file(
        video_path, inference, intent, bias, edge, alert
    )

    # Step 7: Display results
    display_results(inference, intent, bias, alert)

    # Step 8: Plot timeline
    if inference['behavior_events']:
        plot_behavior_timeline(
            inference['behavior_events'],
            inference['duration'],
            title=f'Behavior Timeline — {Path(video_path).name}'
        )

    print(f'\nCase file saved: {case_path}')
    return case_id, case_path, {**inference, 'intent': intent, 'bias': bias,
                                'edge': edge, 'alert': alert}


print('Output & visualization functions ready.')
print()
print('To analyze a video:')
print("  case_id, path, results = analyze_video('/content/drive/MyDrive/test_video.mp4')")
print()
print('To run fast (CPU, every 3rd frame):')
print("  case_id, path, results = analyze_video('/content/test.mp4', frame_step=3)")